# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array} 

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [2]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd
from scipy.linalg import solve_discrete_are

# Load matrices and drop the row-label column
A = pd.read_csv("model-A.tsv", sep="\t").iloc[:, 1:].values
B = pd.read_csv("model-B.tsv", sep="\t").iloc[:, 1:].values
Q = pd.read_csv("cost-Q.tsv", sep="\t").iloc[:, 1:].values
R = pd.read_csv("cost-R.tsv", sep="\t").iloc[:, 1:].values

# Solve the discrete algebraic Riccati equation
P = solve_discrete_are(A, B, Q, R)

# Compute optimal LQR gain
K = np.linalg.inv(R + B.T @ P @ B) @ (B.T @ P @ A)

print(K)

[[0.33445985 1.30445413 1.85813088]]


Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [3]:
# YOUR CHANGES HERE

import pandas as pd
import numpy as np

# K from Part 1
K = np.array([[0.33445985, 1.30445413, 1.85813088]])

# Create dataframe with required column names
df_K = pd.DataFrame(K, columns=["x", "v", "a"])

# Save as TSV
df_K.to_csv("control-K-intended.tsv", sep="\t", index=False)

print(df_K) 

         x         v         a
0  0.33446  1.304454  1.858131


Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [4]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

# Load logged quadcopter data
qc_log = pd.read_csv("qc-log.tsv", sep="\t")

# LQR gain matrix from Part 1
K = np.array([[0.33445985, 1.30445413, 1.85813088]])

# Extract logged states [x, v, a]
X = qc_log[["x", "v", "a"]].values

# Recompute actions using u_t = -K x_t
u_recomputed = -(X @ K.T)

# Save recomputed actions to TSV
df_u = pd.DataFrame(u_recomputed, columns=["u"])
df_u.to_csv("control-u-recomputed.tsv", sep="\t", index=False)

# Show first few rows
print(df_u.head())

          u
0  1.672299
1 -1.152095
2 -1.235183
3 -0.288504
4  0.369202


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [5]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

# Load log data
qc_log = pd.read_csv("qc-log.tsv", sep="\t")

# LQR gain matrix
K = np.array([[0.33445985, 1.30445413, 1.85813088]])

# Extract states
X = qc_log[["x", "v", "a"]].values

# Recompute actions using u = -Kx
u_check = -(X @ K.T).flatten()

# Create output dataframe
df_check = pd.DataFrame({
    "index": qc_log["index"],
    "u_check": u_check
})

# Save to TSV file
df_check.to_csv("qc-check.tsv", sep="\t", index=False)

# Show first rows
print(df_check.head())

   index   u_check
0      0  1.672299
1      1 -1.152095
2      2 -1.235183
3      3 -0.288504
4      4  0.369202


Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [12]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

# Load log data
qc_log = pd.read_csv("qc-log.tsv", sep="\t")

# Extract states and actions
X = qc_log[["x", "v", "a"]].values
U = qc_log["u"].values

# Solve for K using least squares
K_actual = -np.linalg.lstsq(X, U, rcond=None)[0]

print("Estimated control matrix K:")
print(K_actual) 

df_K_actual = pd.DataFrame([K_actual], columns=["x","v","a"])
df_K_actual.to_csv("control-K-actual.tsv", sep="\t", index=False)

print(df_K_actual)

Estimated control matrix K:
[0.34043755 1.30012023 1.95011696]
          x        v         a
0  0.340438  1.30012  1.950117


Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [13]:
# YOUR CHANGES HERE

import pandas as pd

K_actual = [[0.340438, 1.300120, 1.950117]]

df = pd.DataFrame(K_actual, columns=["x","v","a"])
df.to_csv("control-K-actual.tsv", sep="\t", index=False)

print(df)

          x        v         a
0  0.340438  1.30012  1.950117


Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [15]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

# Load training data
qc_train = pd.read_csv("qc-train.tsv", sep="\t")

# Current state x_t and action u_t
X_t = qc_train[["x", "v", "a"]].values[:-1]
U_t = qc_train[["u"]].values[:-1]

# Next state x_{t+1}
X_next = qc_train[["x", "v", "a"]].values[1:]

# Build regression matrix [x_t, v_t, a_t, u_t]
Phi = np.hstack([X_t, U_t])

# Solve least squares problem:
# X_next = Phi @ Theta
Theta, _, _, _ = np.linalg.lstsq(Phi, X_next, rcond=None)

# Extract A and B
A_est = Theta[:3, :].T
B_est = Theta[3, :].reshape(3, 1)

print("Estimated A:")
print(A_est)
print("\nEstimated B:")
print(B_est)

# Save A
df_A = pd.DataFrame(A_est, columns=["x", "v", "a"])
df_A.to_csv("model-A-actual.tsv", sep="\t", index=False)

# Save B
df_B = pd.DataFrame(B_est, columns=["u"])
df_B.to_csv("model-B-actual.tsv", sep="\t", index=False)

print("\nSaved files:")
print("model-A-actual.tsv")
print("model-B-actual.tsv")

Estimated A:
[[ 1.00000000e+00  1.10000000e+00  2.88657986e-15]
 [-1.28195498e-16  9.00000000e-01  9.50000000e-01]
 [-9.31378031e-17  5.55111512e-16  1.10000000e+00]]

Estimated B:
[[-1.11022302e-16]
 [-1.00000000e-02]
 [ 9.00000000e-01]]

Saved files:
model-A-actual.tsv
model-B-actual.tsv


Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [18]:
# YOUR CHANGES HERE

# Clean small floating-point values
A_est = np.round(A_est, 6)
B_est = np.round(B_est, 6)

# Convert to DataFrames
df_A = pd.DataFrame(A_est, columns=["x","v","a"])
df_B = pd.DataFrame(B_est, columns=["u"])

# Save files with required names
df_A.to_csv("model-A-new.tsv", sep="\t", index=False)
df_B.to_csv("model-B-new.tsv", sep="\t", index=False)

# Show results
print(df_A)
print(df_B)

     x    v     a
0  1.0  1.1  0.00
1 -0.0  0.9  0.95
2 -0.0  0.0  1.10
      u
0 -0.00
1 -0.01
2  0.90


Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [20]:
ack_text = """
Discussion with others:
None.

Additional libraries:
None.

Generative AI tools:
I used ChatGPT as a learning aid to help clarify concepts related to linear quadratic regulators, matrix operations, and Python implementation. I also used it to review code structure and debug minor issues during development. All code was written, executed, and tested by me in my notebook, and I verified that the results matched the expected outputs for the assignment. I also referenced the code examples provided in the course materials and example notebooks from class.
"""

# Save to file
with open("acknowledgements.txt", "w") as f:
    f.write(ack_text)

print("Acknowledgements file saved as acknowledgements.txt")

Acknowledgements file saved as acknowledgements.txt
